# Cyber-LLM 7B QLoRA Training on Google Colab (Free T4/L4 GPU)

**Setup:** Runtime → Change runtime type → **GPU (T4/L4)**

**Expected time:** ~2-3 hours on T4, ~1.5 hours on L4

**Output:** 7B GGUF model saved to Google Drive

In [ ]:
# Cell 1: Mount Google Drive (for persistent storage)
from google.colab import drive
drive.mount('/content/drive')

# Create working directory in Drive (persists across sessions)
import os
WORK_DIR = '/content/drive/MyDrive/cyber-llm-training'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f"Working in: {os.getcwd()}")

In [ ]:
# Cell 2: Install dependencies
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes wandb huggingface_hub

# Clone repo
!wget -q https://codeload.github.com/1stsonushaw4590-wq/Own-AI/zip/refs/heads/master -O Own-AI.zip
!unzip -q Own-AI.zip
!mv Own-AI-master Own-AI
%cd Own-AI

# Verify GPU
import torch
print(f"CUDA: {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")

In [ ]:
# Cell 3: Configure secrets
import os

# Add these as Colab Secrets (🔑 icon in left sidebar):
#   WANDB_API_KEY = your_wandb_key
#   HF_TOKEN = your_hf_token

os.environ["WANDB_API_KEY"] = os.getenv("WANDB_API_KEY", "")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN", "")

# Or uncomment and add keys directly (not recommended for shared notebooks):
# os.environ["WANDB_API_KEY"] = "your_wandb_key"
# os.environ["HF_TOKEN"] = "your_hf_token"

print(f"WANDB: {'✅' if os.getenv('WANDB_API_KEY') else '❌'}")
print(f"HF_TOKEN: {'✅' if os.getenv('HF_TOKEN') else '❌'}")

In [ ]:
# Cell 4: Verify dataset
import json, os

DATASET_PATH = "data/cyber_train_merged.jsonl"
if os.path.exists(DATASET_PATH):
    with open(DATASET_PATH) as f:
        lines = f.readlines()
    print(f"✅ Dataset: {len(lines)} samples")
    sample = json.loads(open(DATASET_PATH).readline())
    print(f"Keys: {list(json.loads(open(DATASET_PATH).readline()).keys())}")
else:
    print("❌ Dataset not found - run from repo root")
    !ls -la data/

In [ ]:
# Cell 5: Training configuration
import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments, Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset
from transformers import DataCollatorForSeq2Seq
import wandb

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
DATASET_PATH = "data/cyber_train_merged.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/cyber-llm-training/qwen25-coder-7b-cyber"
MAX_SEQ_LENGTH = 2048
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
MAX_SAMPLES = 0  # 0 = all

# QLoRA 4-bit config for T4 (16GB VRAM)
QLORA_CONFIG = {
    "load_in_4bit": True,
    "bnb_4bit_compute_dtype": torch.bfloat16,
    "bnb_4bit_use_double_quant": True,
    "bnb_4bit_quant_type": "nf4",
}

LORA_CONFIG = {
    "r": 64,
    "lora_alpha": 128,
    "target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    "lora_dropout": 0.05,
    "bias": "none",
    "task_type": "CAUSAL_LM",
}

TRAINING_CONFIG = {
    "output_dir": OUTPUT_DIR,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "num_train_epochs": 3,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.03,
    "logging_steps": 10,
    "save_steps": 500,
    "save_total_limit": 3,
    "bf16": True,
    "tf32": True,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "optim": "paged_adamw_8bit",
    "max_grad_norm": 0.3,
    "max_seq_length": 2048,
    "packing": False,
    "report_to": "wandb" if os.getenv("WANDB_API_KEY") else "none",
    "dataloader_pin_memory": False,
    "dataloader_num_workers": 2,
    "remove_unused_columns": False,
}

In [ ]:
# Cell 5: Training function
import json
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments, Trainer,
    DataCollatorForSeq2Seq
)
from peft import (
    LoraConfig, get_peft_model,
    prepare_model_for_kbit_training, PeftModel
)
from trl import SFTTrainer
import wandb

def format_chat(example):
    if "messages" in example:
        messages = example["messages"]
    elif "instruction" in example and "output" in example:
        messages = [
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": example["output"]}
        ]
    else:
        keys = list(example.keys())
        if len(keys) >= 2:
            messages = [
                {"role": "user", "content": str(example[keys[0]])},
                {"role": "assistant", "content": str(example[keys[1]])}
            ]
        else:
            messages = [{"role": "user", "content": str(example)}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

def train():
    global tokenizer

    # Init wandb
    if os.getenv("WANDB_API_KEY"):
        wandb.init(project="cyber-llm-7b", config={
            "model": "Qwen2.5-Coder-7B",
            "dataset": "cyber-llm-merged",
            "epochs": NUM_EPOCHS,
        })

    # Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME, trust_remote_code=True, padding_side="right"
    )
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    tokenizer.model_max_length = 2048

    # Model with QLoRA
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        ),
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )

    model = prepare_model_for_kbit_training(model)
    model.config.use_cache = False
    model.config.pretraining_tp = 1

    lora_config = LoraConfig(
        r=64, lora_alpha=128,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                       "gate_proj","up_proj","down_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, LoraConfig(
        r=64, lora_alpha=128,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                       "gate_proj","up_proj","down_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    ))
    model.print_trainable_parameters()

    # Load dataset
    dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
    if MAX_SAMPLES > 0:
        dataset = dataset.select(range(MAX_SAMPLES))
    print(f"Dataset: {len(dataset)} samples")

    def format_chat(example):
        if "messages" in example:
            messages = example["messages"]
        elif "instruction" in example and "output" in example:
            messages = [
                {"role": "user", "content": example["instruction"]},
                {"role": "assistant", "content": example["output"]}
            ]
        else:
            keys = list(example.keys())
            messages = [
                {"role": "user", "content": str(example[keys[0]])},
                {"role": "assistant", "content": str(example[keys[1]])}
            ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        return {"text": text}

    dataset = dataset.map(lambda x: format_chat(x), remove_columns=dataset.column_names)

    def tokenize_fn(examples):
    tok = tokenizer(
        examples["text"],
        truncation=True, padding="max_length",
        max_length=2048,
    )
    tok["labels"] = tok["input_ids"].copy()
    return tok

    dataset = dataset.map(tokenize_fn, batched=True, remove_columns=["text"], num_proc=2)
    dataset = dataset.train_test_split(test_size=0.02, seed=42)
    print(f"Train: {len(dataset['train'])}, Eval: {len(dataset['test'])}")

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        args=TrainingArguments(
            output_dir=OUTPUT_DIR,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            num_train_epochs=3,
            learning_rate=2e-4,
            lr_scheduler_type="cosine",
            warmup_ratio=0.03,
            logging_steps=10,
            save_steps=500,
            save_total_limit=3,
            bf16=True, tf32=True,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            optim="paged_adamw_8bit",
            max_grad_norm=0.3,
            max_seq_length=2048,
            packing=False,
            report_to="wandb" if os.getenv("WANDB_API_KEY") else "none",
            dataloader_pin_memory=False,
        ),
        train_dataset=dataset["train"],
        eval_dataset=dataset["test"],
        max_seq_length=2048,
        dataset_text_field="text",
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True, label_pad_token_id=-100),
        packing=False,
    )

    trainer.train()

    # Save adapter
    trainer.save_model(f"{OUTPUT_DIR}/adapter")
    tokenizer.save_pretrained(f"{OUTPUT_DIR}/adapter")
    
    if os.getenv("WANDB_API_KEY"):
        wandb.finish()
    
    return OUTPUT_DIR

In [ ]:
# Cell 6: Execute training
import os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
from datasets import load_dataset
import torch
import json
import wandb

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
DATASET_PATH = "data/cyber_train_merged.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/cyber-llm-training/qwen25-coder-7b-cyber"
MAX_SEQ_LENGTH = 2048
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
MAX_SAMPLES = 0

QLORA_CONFIG = {
    "load_in_4bit": True,
    "bnb_4bit_compute_dtype": torch.bfloat16,
    "bnb_4bit_use_double_quant": True,
    "bnb_4bit_quant_type": "nf4",
}

LORA_CONFIG = {
    "r": 64, "lora_alpha": 128,
    "target_modules": ["q_proj","k_proj","v_proj","o_proj",
                       "gate_proj","up_proj","down_proj"],
    "lora_dropout": 0.05, "bias": "none", "task_type": "CAUSAL_LM"
}

TRAINING_CONFIG = {
    "output_dir": OUTPUT_DIR,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "num_train_epochs": 3,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.03,
    "logging_steps": 10,
    "save_steps": 500,
    "save_total_limit": 3,
    "bf16": True, "tf32": True,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "optim": "paged_adamw_8bit",
    "max_grad_norm": 0.3,
    "max_seq_length": 2048,
    "packing": False,
    "report_to": "wandb" if os.getenv("WANDB_API_KEY") else "none",
    "dataloader_pin_memory": False,
    "dataloader_num_workers": 2,
    "remove_unused_columns": False,
}

import os, torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset
from transformers import DataCollatorForSeq2Seq
import wandb

os.makedirs(OUTPUT_DIR, exist_ok=True)
output_dir = train()
print(f"Training complete: {output_dir}")

In [ ]:
# Cell 7: Merge adapter and quantize to GGUF
!python3 training/merge_and_quantize.py \
  --base Qwen/Qwen2.5-Coder-7B-Instruct \
  --adapter /content/drive/MyDrive/cyber-llm-training/qwen25-coder-7b-cyber/adapter \
  --output /content/drive/MyDrive/cyber-llm-training/qwen25-coder-7b-cyber/merged \
  --gguf /content/drive/MyDrive/cyber-llm-training/qwen25-coder-7b-cyber/gguf \
  --quant q4_k_m

In [ ]:
# Cell 8: Alternative - Manual merge & GGUF conversion
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import os

ADAPTER_PATH = "/content/drive/MyDrive/cyber-llm-training/qwen25-coder-7b-cyber/adapter"
MERGED_DIR = "/content/drive/MyDrive/cyber-llm-training/qwen25-coder-7b-cyber/merged"
GGUF_DIR = "/content/drive/MyDrive/cyber-llm-training/qwen25-coder-7b-cyber/gguf"

os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(GGUF_DIR, exist_ok=True)

print("Loading base model...")
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-Coder-7B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print("Loading adapter...")
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
print("Merging...")
model = model.merge_and_unload()

print("Saving merged model...")
model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct", trust_remote_code=True)
tokenizer.save_pretrained(MERGED_DIR)
print("✅ Merge complete!")

# Note: For GGUF conversion, download merged model and run locally:
# python3 /path/to/llama.cpp/convert_hf_to_gguf.py merged/ --outfile model-f16.gguf --outtype f16
# ./llama-quantize model-f16.gguf model-q4_k_m.gguf q4_k_m

In [ ]:
# Cell 9: Quick inference test (optional)
print("Testing model...")

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "/content/drive/MyDrive/cyber-llm-training/qwen25-coder-7b-cyber/merged"

tokenizer = AutoTokenizer.from_pretrained(
    "/content/drive/MyDrive/cyber-llm-training/qwen25-coder-7b-cyber/merged",
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    "/content/drive/MyDrive/cyber-llm-training/qwen25-coder-7b-cyber/merged",
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

def generate(prompt, max_new_tokens=200):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.2, do_sample=True, top_p=0.9
        )
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("Test: What is SQL injection?")
print(generate("What is SQL injection and how to prevent it?"))

print("\nTest: Nmap scan command")
print(generate("Show nmap command to scan 192.168.1.1"))

## 📋 Colab Training Checklist

### Before Starting:
- [ ] Enable GPU: Runtime → Change runtime type → GPU (T4/L4)
- [ ] Add Secrets (🔑): `WANDB_API_KEY`, `HF_TOKEN`
- [ ] Mount Google Drive (Cell 1)

### Expected Timeline (T4):
- Setup & Install: ~5 min
- Model Loading: ~2 min
- Training (3 epochs, 398 samples): ~2-3 hours
- Merge & Quantize: ~15 min

### Output Location (Google Drive):
```
/content/drive/MyDrive/cyber-llm-training/qwen25-coder-7b-cyber/
├── adapter/          # LoRA adapter (~1GB)
├── merged/           # Merged FP16 model (~14GB)
└── gguf/             # After manual conversion
    └── cyber-llm-q4_k_m.gguf  (~4GB)
```

### After Training:
1. Download GGUF from Google Drive
2. Run locally: `llama-server -m cyber-llm-q4_k_m.gguf --port 8080`
3. Test orchestrator: `./agent/cpp/build/cyber_orchestrator --prompt "scan 127.0.0.1"`